# Field Mapping 04 - Spatial Correlation Analysis

**Report Date:** March 2025  
**Dataset:** Oregon Willamette Valley Agricultural Fields (50 fields)

## Executive Summary

This notebook analyzes spatial correlations between satellite metrics, soil properties, and terrain data for 4 randomly selected fields from the 50-field dataset.

### Key Findings Summary Table

| Section | Finding | Metric |
|---------|---------|--------|
| **7.1 Satellite-Soil** | Correlation between satellite indices and soil properties | r values |
| **7.2 Satellite-Terrain** | Correlation between satellite indices and terrain properties | r values |
| **7.3 Key Findings** | Summary of significant correlations | p < 0.05 |

## Section 1: Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from scipy import stats
import json
import os
import random

# Set random seed for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Configuration Parameters
MIN_PIXEL_COUNT = 30  # Minimum pixels required for valid correlation
SATELLITE_METRICS = ['ndvi', 'msavi', 'evi', 'ndmi']
SOIL_PROPERTIES = ['ph', 'om', 'clay', 'sand', 'cec']  # No drainage
TERRAIN_PROPERTIES = ['elevation', 'slope', 'aspect']

# Define paths
DATA_DIR = 'data/assignment-03'
TERRAIN_DIR = 'data/assignment-04/terrain'
SOIL_DIR = 'data/assignment-04/soil'
SATELLITE_DIR = 'data/assignment-03/satellite'
DOCS_DIR = 'docs/assignment-03'

print('Configuration loaded:')
print(f'  MIN_PIXEL_COUNT: {MIN_PIXEL_COUNT}')
print(f'  SATELLITE_METRICS: {SATELLITE_METRICS}')
print(f'  SOIL_PROPERTIES: {SOIL_PROPERTIES}')
print(f'  TERRAIN_PROPERTIES: {TERRAIN_PROPERTIES}')

## Section 2: Helper Functions

In [ ]:
def load_tiff(path):
    """Load GeoTIFF and return valid pixel values."""
    with rasterio.open(path) as src:
        data = src.read(1)
        if src.nodata is not None:
            valid_mask = (data != src.nodata) & ~np.isnan(data)
        else:
            valid_mask = ~np.isnan(data)
        return data, valid_mask

def calculate_pixel_correlation(array1, array2, min_pixels=MIN_PIXEL_COUNT):
    """Calculate Pearson correlation between two arrays."""
    valid_mask = ~np.isnan(array1) & ~np.isnan(array2)
    valid_count = np.sum(valid_mask)
    if valid_count < min_pixels:
        return np.nan, np.nan, valid_count
    x = array1[valid_mask]
    y = array2[valid_mask]
    r, p = stats.pearsonr(x, y)
    return r, p, valid_count

def load_soil_with_properties(soil_tiff_path, labels_json_path, property_name):
    """Load soil raster and convert label integers to property values."""
    property_map = {'ph': 'ph', 'om': 'om_pct', 'clay': 'clay_pct', 'sand': 'sand_pct', 'cec': 'cec'}
    json_key = property_map.get(property_name, property_name)
    
    with open(labels_json_path, 'r') as f:
        labels = json.load(f)
    
    label_to_value = {}
    for label_id, soil_info in labels.get('soil_definitions', {}).items():
        mukey = soil_info.get('mukey')
        value = soil_info.get(json_key)
        if mukey and value:
            try:
                for lab, muk in labels.get('label_to_mukey', {}).items():
                    if muk == mukey:
                        label_to_value[int(label_id)] = float(value)
            except (ValueError, TypeError):
                pass
    
    with rasterio.open(soil_tiff_path) as src:
        data = src.read(1)
    
    output = np.full_like(data, np.nan, dtype=np.float64)
    data = np.where(data == 0, np.nan, data)
    for label_val, value in label_to_value.items():
        output[data == label_val] = value
    
    return output

print('Helper functions defined.')

## Section 3: Data Loading

In [ ]:
# Load field boundaries
gdf = gpd.read_file(f'{DATA_DIR}/fields_complete.geojson')
print(f'Loaded {len(gdf)} fields')

# Load slope/aspect statistics
slope_df = pd.read_csv(f'{DATA_DIR}/field_slope_aspect.csv')
print(f'Loaded slope/aspect data for {len(slope_df)} fields')

## Section 4: Field Selection

In [ ]:
# Select 4 random fields
selected_fields = random.sample(list(gdf['field_id']), 4)
print(f'Selected fields (seed={RANDOM_SEED}): {selected_fields}')

# Display field info
field_info = gdf[gdf['field_id'].isin(selected_fields)][['field_id', 'area_acres', 'cdl_crop', 'lat', 'lon']]
print('\nField Information:')
print(field_info.to_string(index=False))

## Section 5: Per-Field Data Showcase

For each selected field, we display CDL crop type, soil maps, terrain maps, and satellite maps.

In [ ]:
# Display field information and available data
for field_id in selected_fields:
    print(f'\n=== Field: {field_id} ===')
    field_row = gdf[gdf['field_id'] == field_id].iloc[0]
    print(f'  Crop: {field_row["cdl_crop"]}')
    print(f'  Area: {field_row["area_acres"]:.1f} acres')
    print(f'  Location: ({field_row["lat"]:.4f}, {field_row["lon"]:.4f})')
    
    # Check available data files
    print(f'  Satellite: {len([m for m in SATELLITE_METRICS if os.path.exists(f"{SATELLITE_DIR}/{field_id}_{m}.tif")])} files')
    print(f'  Terrain: {len([p for p in TERRAIN_PROPERTIES if os.path.exists(f"{TERRAIN_DIR}/{field_id}_{p}.tif")])} files')
    print(f'  Soil: {"Yes" if os.path.exists(f"{SOIL_DIR}/{field_id}_soil.tif") else "No"}')

## Section 6: Per-Field Histograms

Histograms showing distribution of satellite and terrain values for each field.

In [ ]:
# Generate histograms for each field
for field_id in selected_fields:
    print(f'\n=== {field_id} Histograms ===')
    
    # Satellite histograms
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    fig.suptitle(f'{field_id} - Satellite Metrics', fontsize=14)
    
    for idx, metric in enumerate(SATELLITE_METRICS):
        ax = axes[idx // 2, idx % 2]
        sat_path = f'{SATELLITE_DIR}/{field_id}_{metric}.tif'
        if os.path.exists(sat_path):
            data, mask = load_tiff(sat_path)
            valid_data = data[mask]
            ax.hist(valid_data, bins=30, edgecolor='black', alpha=0.7)
            ax.set_title(f'{metric.upper()}')
            ax.set_xlabel('Value')
            ax.set_ylabel('Frequency')
            ax.text(0.02, 0.98, f'n={len(valid_data)}\nmean={np.mean(valid_data):.3f}', 
                    transform=ax.transAxes, fontsize=8, verticalalignment='top')
    plt.tight_layout()
    plt.show()
    
    # Terrain histograms
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    fig.suptitle(f'{field_id} - Terrain Properties', fontsize=14)
    
    for idx, prop in enumerate(TERRAIN_PROPERTIES):
        ax = axes[idx]
        terr_path = f'{TERRAIN_DIR}/{field_id}_{prop}.tif'
        if os.path.exists(terr_path):
            data, mask = load_tiff(terr_path)
            valid_data = data[mask]
            ax.hist(valid_data, bins=30, edgecolor='black', alpha=0.7, color='green')
            ax.set_title(f'{prop.capitalize()}')
            ax.set_xlabel('Value')
            ax.set_ylabel('Frequency')
            ax.text(0.02, 0.98, f'n={len(valid_data)}\nmean={np.mean(valid_data):.2f}', 
                    transform=ax.transAxes, fontsize=8, verticalalignment='top')
    plt.tight_layout()
    plt.show()

## Section 7: Spatial Correlation Analysis

Analyze pixel-by-pixel correlations between satellite metrics and soil/terrain properties.

In [ ]:
# Placeholder - will be populated after testing
print('Correlation analysis sections will be added.')

## Section 7.1: Satellite vs Soil Correlations

In [ ]:
# Satellite vs Soil Correlations
results_soil = []

for field_id in selected_fields:
    print(f'Processing {field_id}...')
    
    for sat_metric in SATELLITE_METRICS:
        sat_path = f'{SATELLITE_DIR}/{field_id}_{sat_metric}.tif'
        if not os.path.exists(sat_path):
            continue
        
        sat_data, sat_mask = load_tiff(sat_path)
        
        for soil_prop in SOIL_PROPERTIES:
            soil_tiff_path = f'{SOIL_DIR}/{field_id}_soil.tif'
            soil_labels_path = f'{SOIL_DIR}/{field_id}_soil_labels.json'
            
            if not os.path.exists(soil_tiff_path):
                continue
            
            soil_data = load_soil_with_properties(soil_tiff_path, soil_labels_path, soil_prop)
            
            # Match shapes by using minimum dimensions
            min_h = min(sat_data.shape[0], soil_data.shape[0])
            min_w = min(sat_data.shape[1], soil_data.shape[1])
            
            sat_crop = sat_data[:min_h, :min_w]
            soil_crop = soil_data[:min_h, :min_w]
            
            r, p, n = calculate_pixel_correlation(sat_crop, soil_crop)
            
            results_soil.append({
                'field_id': field_id,
                'satellite': sat_metric,
                'soil_property': soil_prop,
                'r': r,
                'p': p,
                'n_pixels': n,
                'significant': 'Y' if p < 0.05 else 'N'
            })

df_soil = pd.DataFrame(results_soil)
print('\n=== Satellite vs Soil Correlations ===')
print(df_soil.to_string(index=False))

## Section 7.2: Satellite vs Terrain Correlations

In [ ]:
# Satellite vs Terrain Correlations
results_terrain = []

for field_id in selected_fields:
    print(f'Processing {field_id}...')
    
    for sat_metric in SATELLITE_METRICS:
        sat_path = f'{SATELLITE_DIR}/{field_id}_{sat_metric}.tif'
        if not os.path.exists(sat_path):
            continue
        
        sat_data, sat_mask = load_tiff(sat_path)
        
        for terr_prop in TERRAIN_PROPERTIES:
            terr_path = f'{TERRAIN_DIR}/{field_id}_{terr_prop}.tif'
            
            if not os.path.exists(terr_path):
                continue
            
            terr_data, terr_mask = load_tiff(terr_path)
            
            # Match shapes
            min_h = min(sat_data.shape[0], terr_data.shape[0])
            min_w = min(sat_data.shape[1], terr_data.shape[1])
            
            sat_crop = sat_data[:min_h, :min_w]
            terr_crop = terr_data[:min_h, :min_w]
            
            r, p, n = calculate_pixel_correlation(sat_crop, terr_crop)
            
            results_terrain.append({
                'field_id': field_id,
                'satellite': sat_metric,
                'terrain_property': terr_prop,
                'r': r,
                'p': p,
                'n_pixels': n,
                'significant': 'Y' if p < 0.05 else 'N'
            })

df_terrain = pd.DataFrame(results_terrain)
print('\n=== Satellite vs Terrain Correlations ===')
print(df_terrain.to_string(index=False))

## Section 7.3: Key Findings

In [ ]:
# Summary of significant correlations
print('=== Summary: Significant Correlations (p < 0.05) ===\n')

# Soil correlations
sig_soil = df_soil[df_soil['significant'] == 'Y']
print(f'Satellite-Soil: {len(sig_soil)} significant correlations')
if len(sig_soil) > 0:
    print(sig_soil.to_string(index=False))

# Terrain correlations
sig_terrain = df_terrain[df_terrain['significant'] == 'Y']
print(f'\nSatellite-Terrain: {len(sig_terrain)} significant correlations')
if len(sig_terrain) > 0:
    print(sig_terrain.to_string(index=False))

# Mean correlations
print('\n=== Mean Correlations Across Fields ===')
print('\nSoil:')
print(df_soil.groupby('soil_property')['r'].mean())
print('\nTerrain:')
print(df_terrain.groupby('terrain_property')['r'].mean())